# Face Matching with `face_recognition`

This notebook demonstrates the face-matching component prepared for **La2eny**. It builds a small gallery of face encodings and compares a query image against that gallery.

> Use only consented images. A match is a candidate for human review, never proof of identity.

## 1. Install dependencies

Google Colab is recommended because `dlib` installation can be difficult on some local systems.

In [ ]:
!pip install -q face-recognition opencv-python matplotlib

## 2. Imports

In [ ]:
from pathlib import Path
import face_recognition
import cv2
import matplotlib.pyplot as plt

## 3. Build a face gallery

Each reference image should contain one clearly visible face. The filename stem is used as its label.

In [ ]:
def build_face_gallery(gallery_dir: str):
    gallery_path = Path(gallery_dir)
    encodings, labels = [], []

    for image_path in sorted(gallery_path.glob('*')):
        if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            continue

        image = face_recognition.load_image_file(str(image_path))
        face_encodings = face_recognition.face_encodings(image)
        if not face_encodings:
            print(f'Skipping image without a detectable face: {image_path.name}')
            continue

        encodings.append(face_encodings[0])
        labels.append(image_path.stem)

    return encodings, labels

## 4. Match a query face

The default tolerance is the library's commonly used starting point. It should be calibrated using representative validation images before any real-world use.

In [ ]:
def match_face(query_path: str, known_encodings, known_labels, tolerance: float = 0.6):
    query_image = face_recognition.load_image_file(query_path)
    query_encodings = face_recognition.face_encodings(query_image)

    if not query_encodings:
        raise ValueError('No face was detected in the query image.')
    if not known_encodings:
        raise ValueError('The reference gallery is empty.')

    query_encoding = query_encodings[0]
    distances = face_recognition.face_distance(known_encodings, query_encoding)
    best_index = int(distances.argmin())
    is_match = bool(distances[best_index] <= tolerance)

    return {
        'matched': is_match,
        'label': known_labels[best_index] if is_match else 'Unknown',
        'distance': float(distances[best_index]),
    }

## 5. Run with your own images

Expected layout:

```text
data/
├── known_faces/
│   ├── reference_01.jpg
│   └── reference_02.jpg
└── query_face.jpg
```

In [ ]:
# Update these paths to images you have permission to use.
# gallery_encodings, gallery_labels = build_face_gallery('data/known_faces')
# result = match_face('data/query_face.jpg', gallery_encodings, gallery_labels)
# print(result)

## Notes

- The gallery assumes one usable face per reference image.
- Lighting, pose, image quality, and demographic variation can affect results.
- Evaluate false-acceptance and false-rejection rates before considering deployment.
- Keep identity images private and access-controlled.